# 6장 예제 — 6.11 공정능력지수 `필수`

- **교재 출처**: manuscript/61_ch6_최적화.md
- **실행 방법**: example 폴더에서  python "6장/<이 파일>" (교재 코드의 data/ 상대경로를 그대로 쓰기 위해 작업 디렉터리는 example)
- **선행 필요**: 이 절의 코드는 앞 절에서 만든 변수를 이어 쓴다. 단독 실행이 필요하면 같은 장의 _장전체_실행본.py 를 쓰거나 아래 절을 먼저 실행하라 — 07_6-8_불량_발생_패턴_분석.py, 08_6-9_원인_추적_기법.py, 09_6-10_SPC_관리도.py
- **실행 상태**: 단독 실행 확인됨(이 파일만 돌려도 끝까지 실행된다).

In [ ]:
# [환경 설정] data/ 경로 자동 감지
import os
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')

In [ ]:
# --- 실행 준비 (교재에서는 앞 절에서 이미 실행한 코드) --------------------
import pandas as pd
import numpy as np
df = pd.read_csv("data/battery_process_data.csv")
df["건조로_1구간_온도"] = df["건조로_1구간_온도"].replace(9999, np.nan)
df = df.fillna(df.median(numeric_only=True))   # 3.7절 clean_data() 규약

### 실습 데이터의 Cp/Cpk 산출

In [ ]:
specs = {  # (LSL, USL)
    "믹싱_RPM":        (1650, 1950),
    "믹싱_온도":        (22, 28),
    "코팅_토출압력":     (105, 135),
    "건조로_1구간_온도":  (100, 130),
    "프레스_압력":       (35, 55),
    "프레스_Gap":       (85, 105),
}

def capability(s, lsl, usl):
    mu, sd = s.mean(), s.std()
    cp = (usl - lsl) / (6 * sd)
    cpk = min(usl - mu, mu - lsl) / (3 * sd)
    return mu, sd, cp, cpk

print(f"{'변수':14s} {'평균':>8s} {'표준편차':>8s} {'Cp':>6s} {'Cpk':>6s}")
for col, (l, u) in specs.items():
    mu, sd, cp, cpk = capability(df[col], l, u)
    print(f"{col:14s} {mu:8.2f} {sd:8.2f} {cp:6.2f} {cpk:6.2f}")

# 교재 실행 결과 ------------------------------------------------------
#   변수                   평균     표준편차     Cp    Cpk
#   믹싱_RPM          1747.57    79.04   0.63   0.41
#   믹싱_온도             24.88     1.52   0.66   0.63
#   코팅_토출압력          120.27     8.15   0.61   0.60
#   건조로_1구간_온도       109.99     3.91   1.28   0.85
#   프레스_압력            39.91     3.98   0.84   0.41
#   프레스_Gap           95.32     6.09   0.55   0.53